# NB_00_Generate_Data
Simulates the source systems of **Calder & Wren Imports Ltd**, a fictional UK homeware importer.

Every run drops new CSV files into `Files/landing/<entity>/batch=<date>/`, exactly like a nightly
export from an ERP, a freight forwarder portal and a bank feed would.

* **Deterministic** – the whole business is replayed from `history_start` with a fixed seed, so the
  same day always produces the same records.
* **Incremental** – only events after the stored watermark are written. The first run back-fills
  history in monthly batches; later runs add one batch per run.
* **Deliberately messy** – duplicates, mixed date formats, inconsistent casing, orphan keys,
  late-arriving invoices and shipment rows that are re-sent when their status changes.

In [ ]:
as_of = ""                      # YYYY-MM-DD, blank = today
history_start = "2025-01-01"
seed = 42
landing_local = "/lakehouse/default/Files/landing"   # default lakehouse mount

In [ ]:
import json, os, math
from datetime import date, timedelta
import numpy as np
import pandas as pd

AS_OF = date.fromisoformat(as_of) if as_of else date.today()
START = date.fromisoformat(history_start)
STATE_PATH = os.path.join(landing_local, "_state.json")

watermark = START - timedelta(days=1)
if os.path.exists(STATE_PATH):
    with open(STATE_PATH) as f:
        watermark = date.fromisoformat(json.load(f)["watermark"])
print(f"Emitting events after {watermark} up to {AS_OF}")

## Master data

In [ ]:
rng_m = np.random.default_rng(seed)

COUNTRIES = {  # currency, sea transit days, freight per carton (GBP)
    "India":    ("USD", 28, 6.0),
    "Vietnam":  ("USD", 35, 7.5),
    "China":    ("USD", 38, 8.0),
    "Turkey":   ("EUR", 14, 3.5),
    "Portugal": ("EUR", 7,  2.0),
}
SUPPLIER_NAMES = [
    ("Jaipur Craft Exports", "India"), ("Moradabad Metalworks", "India"), ("Karur Home Textiles", "India"),
    ("Saigon Bamboo Co", "Vietnam"), ("Binh Duong Ceramics", "Vietnam"),
    ("Ningbo Bright Housewares", "China"), ("Foshan Lumen Lighting", "China"), ("Yiwu Storage Solutions", "China"),
    ("Denizli Tekstil", "Turkey"), ("Kutahya Porselen", "Turkey"),
    ("Aveiro Stoneware", "Portugal"), ("Guimaraes Linen Mill", "Portugal"),
]
suppliers = []
for i, (name, country) in enumerate(SUPPLIER_NAMES, 1):
    ccy, transit, freight = COUNTRIES[country]
    suppliers.append(dict(
        supplier_id=f"SUP-{i:03d}", supplier_name=name, country=country, currency=ccy,
        payment_terms_days=int(rng_m.choice([30, 45, 60])),
        _transit=transit, _freight=freight,
        _delay_mean=float(rng_m.choice([0.5, 1.5, 4.0, 9.0], p=[0.3, 0.35, 0.2, 0.15])),  # hidden reliability
        _short_ship=float(rng_m.choice([0.03, 0.08, 0.20], p=[0.5, 0.3, 0.2])),
    ))

CATEGORIES = {  # (duty rate, typical unit cost in USD, units per carton)
    "Cookware": (0.06, 9.0, 6), "Tableware": (0.08, 3.5, 24), "Textiles": (0.12, 5.0, 20),
    "Storage": (0.04, 4.0, 12), "Garden": (0.03, 11.0, 4), "Lighting": (0.047, 14.0, 6),
}
ADJ = ["Oak", "Slate", "Linen", "Copper", "Sage", "Coastal", "Heritage", "Nordic", "Amber", "Willow"]
NOUN = {
    "Cookware": ["Casserole", "Saute Pan", "Stockpot", "Roasting Tin"],
    "Tableware": ["Dinner Plate", "Mug", "Pasta Bowl", "Serving Platter"],
    "Textiles": ["Throw", "Cushion Cover", "Tea Towel Set", "Bath Sheet"],
    "Storage": ["Basket", "Glass Jar", "Shelf Crate", "Laundry Hamper"],
    "Garden": ["Planter", "Lantern", "Bistro Chair", "Fire Bowl"],
    "Lighting": ["Table Lamp", "Pendant", "Floor Lamp", "Wall Light"],
}
FX0 = {"USD": 0.79, "EUR": 0.84}
products = []
for i in range(1, 61):
    cat = list(CATEGORIES)[i % 6]
    duty, base_cost, upc = CATEGORIES[cat]
    sup = suppliers[int(rng_m.integers(0, len(suppliers)))]
    cost_usd = base_cost * float(rng_m.uniform(0.6, 1.7))
    cost_ccy = round(cost_usd * FX0["USD"] / FX0[sup["currency"]], 2)
    products.append(dict(
        sku=f"SKU-{i:04d}",
        product_name=f"{ADJ[int(rng_m.integers(0, 10))]} {NOUN[cat][int(rng_m.integers(0, 4))]}",
        category=cat, supplier_id=sup["supplier_id"], unit_cost_ccy=cost_ccy,
        list_price_gbp=round(cost_usd * FX0["USD"] * float(rng_m.uniform(1.7, 2.7)), 2),
        units_per_carton=upc, _duty=duty, _pop=float(rng_m.pareto(1.6) + 0.2),
    ))
SUP = {s["supplier_id"]: s for s in suppliers}

C1 = ["Hartley", "Pemberton", "Ashdown", "Mercer", "Thistle", "Brambles", "Kingsmead", "Fenwick", "Larkfield",
      "Oakhurst", "Marlow", "Redfern", "Stanton", "Wetherby", "Calloway", "Dunmore", "Elmstead", "Foxglove"]
C2 = ["Home", "Interiors", "Living", "Garden Centre", "& Sons", "Homestore", "Lifestyle", "Trading", "& Co", "Kitchens"]
SEGMENTS = {"Independent": 0.0, "Regional Chain": 0.08, "Online": 0.05, "Garden Centre": 0.03}
REGIONS = ["London", "South East", "South West", "Midlands", "North West", "Yorkshire", "Scotland", "Wales"]
N_CUSTOMERS = 110
customers, seen = [], set()
while len(customers) < N_CUSTOMERS:
    name = f"{C1[int(rng_m.integers(0, len(C1)))]} {C2[int(rng_m.integers(0, len(C2)))]} Ltd"
    if name in seen:
        continue
    seen.add(name)
    i = len(customers) + 1
    seg = str(rng_m.choice(list(SEGMENTS), p=[0.45, 0.2, 0.2, 0.15]))
    # first 80 customers exist from day one, the rest are won over time
    created = START if i <= 80 else START + timedelta(days=int(rng_m.integers(20, 900)))
    customers.append(dict(
        customer_id=f"CUS-{i:04d}", customer_name=name, segment=seg,
        region=str(rng_m.choice(REGIONS)), credit_terms_days=int(rng_m.choice([30, 45, 60], p=[0.6, 0.25, 0.15])),
        credit_limit_gbp=int(rng_m.choice([5000, 10000, 25000, 50000])), created_date=created.isoformat(),
        _size=float(rng_m.pareto(1.3) + 0.3) * (3 if seg == "Regional Chain" else 1),
        _late_mean=float(rng_m.choice([0.0, 3.0, 9.0, 25.0], p=[0.35, 0.35, 0.2, 0.1])),      # hidden payment habit
        _ref_style=str(rng_m.choice(["clean", "messy", "none"], p=[0.65, 0.2, 0.15])),          # hidden remittance habit
        _alias=bool(rng_m.random() < 0.06),   # pays from a differently named account
    ))

## Replay the business day by day
Each purchase order and invoice decides its whole future (shipment, arrival, payment) when it is
created, and every resulting row is tagged with the date it becomes visible (`_emit`).

In [ ]:
rng_fx, rng_po, rng_so, rng_noise = (np.random.default_rng(seed + k) for k in (1, 2, 3, 4))

def next_weekday(d):
    while d.weekday() >= 5:
        d += timedelta(days=1)
    return d

def payer_variant(name, rng):
    v = name
    r = rng.random()
    if r < 0.35:
        v = v.upper()
    elif r < 0.55:
        v = v.replace("Ltd", "Limited")
    elif r < 0.70:
        v = v.upper().replace("LTD", "LTD.")
    elif r < 0.80:
        v = v.replace("&", "and")
    return v

def reference_text(inv_no, style, rng):
    n = int(inv_no.split("-")[1])
    if style == "none":
        return str(rng.choice(["", "PAYMENT", "BACS", "SUPPLIER PAYMENT"]))
    if style == "messy":
        return str(rng.choice([f"inv {n}", f"Payment INV{n:06d} thanks", f"INV_{n}", f"calder wren inv#{n:06d}", f"Inv-{n:05d} BACS"]))
    return inv_no

fx, rows = dict(FX0), {k: [] for k in
                       ["fx_rates", "purchase_orders", "shipments", "invoice_lines", "bank_transactions"]}
po_n = shp_n = inv_n = txn_n = 0
prod_p = np.array([p["_pop"] for p in products]); prod_p /= prod_p.sum()
PORTS = ["Felixstowe", "Southampton", "London Gateway"]

d = START
while d <= AS_OF:
    iso = d.isoformat()
    # --- FX (random walk) ---
    for c in fx:
        fx[c] *= math.exp(float(rng_fx.normal(0, 0.004)))
        rows["fx_rates"].append(dict(rate_date=iso, currency=c, rate_to_gbp=round(fx[c], 5), _emit=d))

    if d.weekday() < 5:
        years = (d - START).days / 365.0
        # --- Purchase orders and their shipments ---
        for _ in range(int(rng_po.poisson(5))):
            po_n += 1
            p = products[int(rng_po.choice(len(products), p=prod_p))]
            s = SUP[p["supplier_id"]]
            cartons = int(rng_po.integers(20, 200))
            qty = cartons * p["units_per_carton"]
            cost = round(p["unit_cost_ccy"] * (1 + 0.04 * years) * (1 + float(rng_po.normal(0, 0.03))), 2)
            exp_ship = d + timedelta(days=int(rng_po.integers(14, 31)))
            dirt = rng_po.random(3)
            po = dict(po_id=f"PO-{po_n:06d}", po_date=iso, supplier_id=s["supplier_id"], sku=p["sku"],
                      qty_ordered="" if dirt[0] < 0.003 else qty, unit_cost=cost,
                      currency=(f" {s['currency'].lower()} " if dirt[1] < 0.03 else s["currency"]),
                      expected_ship_date=exp_ship.isoformat(), _emit=d)
            rows["purchase_orders"].append(po)
            if dirt[2] < 0.01:
                rows["purchase_orders"].append(dict(po))          # exact duplicate

            shp_n += 1
            ship = exp_ship + timedelta(days=int(round(float(rng_po.exponential(s["_delay_mean"])))) - 1)
            eta = ship + timedelta(days=s["_transit"])
            congestion = 2.5 if eta.month in (11, 12, 1) else 1.0
            arrive = eta + timedelta(days=int(round(float(rng_po.exponential(1.2 * congestion)) + float(rng_po.normal(0, 1.5)))))
            arrive = max(arrive, ship + timedelta(days=3))
            received = qty if rng_po.random() > s["_short_ship"] else int(qty * float(rng_po.uniform(0.82, 0.98)))
            surge = 1.6 if date(2025, 12, 1) <= ship <= date(2026, 3, 31) else 1.0     # freight rate spike
            freight = round(s["_freight"] * cartons * surge * float(rng_po.uniform(0.85, 1.2)), 2)
            duty = round(received * cost * fx[s["currency"]] * p["_duty"], 2)
            base = dict(shipment_id=f"SHP-{shp_n:06d}", po_id=po["po_id"],
                        container_no=f"{rng_po.choice(['MSKU', 'CMAU', 'HLXU', 'OOLU'])}{int(rng_po.integers(1000000, 9999999))}",
                        ship_date=ship.isoformat(), eta_date=eta.isoformat(), port=str(rng_po.choice(PORTS)))
            rows["shipments"].append(dict(base, arrival_date="", qty_received="", freight_cost_gbp=freight,
                                          duty_gbp="", status="In Transit", _emit=ship))
            rows["shipments"].append(dict(base, arrival_date=arrive.isoformat(), qty_received=received,
                                          freight_cost_gbp=freight, duty_gbp=duty, status="Arrived", _emit=arrive))

        # --- Sales invoices and the bank receipts that settle them ---
        live = [c for c in customers if c["created_date"] <= iso]
        w = np.array([c["_size"] for c in live]); w /= w.sum()
        season = 1.3 if d.month in (10, 11, 12) else (0.85 if d.month in (1, 2) else 1.0)
        for _ in range(int(rng_so.poisson(34 * season * (1 + 0.15 * years)))):
            inv_n += 1
            c = live[int(rng_so.choice(len(live), p=w))]
            inv_no = f"INV-{inv_n:06d}"
            due = d + timedelta(days=c["credit_terms_days"])
            emit = d + timedelta(days=int(rng_so.integers(1, 5))) if rng_so.random() < 0.05 else d   # late-arriving
            total = 0.0
            for line_no in range(1, int(rng_so.integers(1, 7)) + 1):
                p = products[int(rng_so.choice(len(products), p=prod_p))]
                qty = int(rng_so.integers(1, 9)) * p["units_per_carton"]
                price = round(p["list_price_gbp"] * (1 + 0.03 * years) * (1 - SEGMENTS[c["segment"]]), 2)
                disc = int(rng_so.choice([0, 5, 10], p=[0.7, 0.2, 0.1]))
                dirt = rng_so.random(4)
                bad = dirt[0] < 0.004      # orphan SKU or non-positive quantity: must be rejected downstream
                line = dict(invoice_id=inv_no, line_no=line_no, invoice_date=iso,
                            customer_id=(c["customer_id"].lower() + " " if dirt[1] < 0.02 else c["customer_id"]),
                            sku=("SKU-9999" if bad and dirt[2] < 0.5 else p["sku"]),
                            qty=(-qty if bad and dirt[2] >= 0.5 else qty),
                            unit_price_gbp=price, discount_pct=disc, due_date=due.isoformat(), _emit=emit)
                rows["invoice_lines"].append(line)
                if dirt[3] < 0.01:
                    rows["invoice_lines"].append(dict(line))
                if not bad:
                    total += round(qty * price * (1 - disc / 100), 2)
            total = round(total, 2)
            outcome = rng_so.random()
            if total <= 0 or outcome < 0.04:
                continue                                           # never paid
            paid = round(total * float(rng_so.uniform(0.9, 0.98)), 2) if outcome < 0.10 else total   # short-pay
            pay = next_weekday(due + timedelta(days=int(round(float(rng_so.exponential(c["_late_mean"] + 0.01)))) - int(rng_so.integers(0, 6))))
            pay = max(pay, next_weekday(d + timedelta(days=2)))
            txn_n += 1
            dirt = rng_so.random(3)
            txn = dict(txn_id=f"BK{txn_n:08d}",
                       txn_date=(pay.strftime("%d/%m/%Y") if dirt[0] < 0.15 else pay.isoformat()),
                       amount_gbp=(f"{paid:,.2f}" if dirt[1] < 0.10 else f"{paid:.2f}"),
                       payer_name=(f"{c['customer_name'].split()[0]} Holdings Group" if c["_alias"]
                                   else payer_variant(c["customer_name"], rng_so)),
                       reference=reference_text(inv_no, c["_ref_style"], rng_so),
                       _emit=pay, _truth=inv_no)
            rows["bank_transactions"].append(txn)
            if dirt[2] < 0.007:
                rows["bank_transactions"].append(dict(txn))

        # --- Bank noise that is not a customer receipt ---
        if rng_noise.random() < 0.3:
            txn_n += 1
            who, ref = [("HMRC", "VAT REPAYMENT"), ("STRIPE PAYMENTS UK", "STRIPE PAYOUT"), ("BARCLAYS", "INTEREST"),
                        ("J SMITH", "refund"), ("UNKNOWN PAYER", "")][int(rng_noise.integers(0, 5))]
            rows["bank_transactions"].append(dict(txn_id=f"BK{txn_n:08d}", txn_date=iso,
                                                  amount_gbp=f"{float(rng_noise.uniform(20, 4000)):.2f}",
                                                  payer_name=who, reference=ref, _emit=d, _truth=""))
    d += timedelta(days=1)

rows["suppliers"] = [dict(s, _emit=START) for s in suppliers]
rows["products"] = [dict(p, _emit=START) for p in products]
rows["customers"] = [dict(c, _emit=date.fromisoformat(c["created_date"])) for c in customers]

## Write only the new events, one folder per batch

In [ ]:
def batch_edges(after, upto):
    """Month-end cut-offs between the watermark and as_of (a single batch for a daily run)."""
    edges, cur = [], after + timedelta(days=1)
    while cur <= upto:
        month_end = (cur.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
        edges.append(min(month_end, upto))
        cur = edges[-1] + timedelta(days=1)
    return edges

written, lo = 0, watermark
for hi in batch_edges(watermark, AS_OF):
    for entity, recs in rows.items():
        out = [{k: v for k, v in r.items() if not k.startswith("_")} for r in recs if lo < r["_emit"] <= hi]
        if not out:
            continue
        folder = os.path.join(landing_local, entity, f"batch={hi.isoformat()}")
        os.makedirs(folder, exist_ok=True)
        pd.DataFrame(out).to_csv(os.path.join(folder, f"{entity}.csv"), index=False)
        written += len(out)
    lo = hi

os.makedirs(landing_local, exist_ok=True)
with open(STATE_PATH, "w") as f:
    json.dump({"watermark": AS_OF.isoformat()}, f)
print(f"Wrote {written:,} rows. Watermark is now {AS_OF}")